# Notebook 06 — recording support and training loss

A **prospectively specified exploratory follow-up informed by Notebook 05**, not independent preregistered confirmation. The [frozen specification](../reports/hr_support_loss_specification.md) precedes new comparisons. Read the [report](../reports/hr_support_loss/838b9efbf4bfd731/report.md) and its aggregate appendices.

Questions: Does HR help with denser, more recent recordings? Does median-oriented linear regression improve MAE relative to Ridge? Timing remains provisional; this is not validated real-time prediction. All 32 final-test participants remain sealed. This notebook displays only verified aggregates and synthetic examples.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from garmin_preprocessing.common import verify_bundle
from garmin_hr_support_loss.support import accepted_occupancy

BUNDLE = ROOT / "private/hr_support_loss/runs/c9caf911d30a6271"
REPORT = ROOT / "reports/hr_support_loss/838b9efbf4bfd731"
manifest = verify_bundle(BUNDLE)
verify_bundle(REPORT)
assert manifest["preserved_ema_rows"] == 3706
print("Verified unchanged development EMA accounting:", manifest["preserved_ema_rows"])
print(json.loads((REPORT / "occupancy_verification.json").read_text()))

## Fixed support policies and a synthetic boundary example

A retains Notebook 05's complete-case HR cohort. B adds ≥80% accepted-reading bin occupancy. C adds freshness ≤60 seconds. These are provisional choices, not device standards. Bins are [left,right), anchored to cutoff minus lookback. Existing diagnostics were independently verified to count accepted readings.

Four occupied bins out of five can still contain a two-minute gap. The following timestamps are synthetic; no beat/waveform or missing observation is reconstructed.

In [ ]:
synthetic_times = np.array([0., 60., 120., 240., 300.])
synthetic_cutoff = 300.
synthetic_accepted = synthetic_times[synthetic_times < synthetic_cutoff]
occupied = accepted_occupancy(synthetic_times, 0., 5)
freshness = synthetic_cutoff - synthetic_accepted[-1]
assert occupied == 4 and freshness == 60
print({"synthetic_occupied_bins": occupied, "possible_bins": 5,
       "policy_B_passes": 5 * occupied >= 4 * 5,
       "policy_C_passes": freshness <= 60,
       "maximum_internal_gap_seconds": float(np.diff(synthetic_accepted).max())})
print("The cutoff reading is excluded. Occupied bins do not establish continuous recording.")

## Coverage and population changes

The common cohort intersects all eight lookbacks; training and validation both use that intersection for refits. B/C common cohorts contain only 22 responses from seven participants and are descriptive only. No folds were redrawn or thresholds relaxed. The composition appendix reports retained/excluded survey and PSS distributions. All 3,706 original responses remain accounted for.

In [ ]:
cohorts = pd.read_json(REPORT / "cohorts.json")
folds = pd.read_json(REPORT / "fold_counts.json")
display(cohorts.loc[cohorts.scope.eq("native")])
display(folds.loc[folds.stage.eq("support_refit") & folds.scope.eq("common")
                  & folds.policy.eq("C") & folds.lookback_minutes.eq(5)])
display(Image(filename=str(REPORT / "retention.png")))

## Keep saved-prediction filtering separate from retraining

`saved_subset` evaluates Notebook 05 predictions on restricted validation populations, preserving their original training cohort. `support_refit` fits all matching dummy/context/HR Ridge models inside the new eligible cohort. `retraining_intervals.json` compares these stages on exactly shared validation responses.

MAE is primary; RMSE and participant-averaged MAE are also recorded, pooled and per fold. Intervals use 2,000 paired participant-cluster resamples of fixed predictions, not model-refitting uncertainty or multiplicity-adjusted confirmation.

In [ ]:
metrics = pd.read_json(REPORT / "metrics.json")
intervals = pd.read_json(REPORT / "paired_intervals.json")
display(metrics.loc[metrics.scope.eq("native") & metrics.fold.eq("pooled")
                    & metrics.survey_category.eq("all") & metrics.policy.eq("C")
                    & metrics.model.isin(["median", "context", "context_hr"]),
                    ["stage", "lookback_minutes", "model", "responses", "participants", "mae", "rmse", "participant_mae"]])
display(Image(filename=str(REPORT / "support_comparisons.png")))
display(Image(filename=str(REPORT / "recording_diagnostics.png")))

## Median loss on unchanged A cohorts only

QuantileRegressor(quantile=.5, alpha=.01, solver='highs') uses training-only scaling/encoding. Its L1/average-pinball penalty is not equivalent to Ridge's L2/squared-loss alpha=1. No quantile fits use B/C, and no alpha was tuned.

All 240 fits converged; 234 have constant validation predictions equal to the matching training median. This is a result, not a reason to retune. Variation and convergence checks are saved. Lower MAE than Ridge generally reproduces the median dummy rather than revealing HR information.

In [ ]:
display(metrics.loc[metrics.stage.eq("loss_only") & metrics.scope.eq("native")
                    & metrics.fold.eq("pooled") & metrics.survey_category.eq("all")
                    & metrics.model.isin(["median", "context_hr", "q_context_hr"]),
                    ["lookback_minutes", "model", "mae", "rmse", "participant_mae"]])
display(intervals.loc[intervals.stage.eq("loss_only") & intervals.scope.eq("native")
                      & intervals.model.eq("q_context_hr") & intervals.baseline.eq("median")])
display(Image(filename=str(REPORT / "loss_comparison.png")))

## Survey reference periods and interpretation

Morning asks since waking; Midday/Endday ask since the last prompt. We do not know whether a prompt means scheduled, delivered, opened or completed. No waking or preceding-prompt times are invented from previous completed surveys.

C at 12 hours has a small favorable native HR point difference versus median, but its interval includes no improvement and it loses 91.9% of the A responses. Common B/C results involve seven participants. These comparisons do not identify an optimal lookback or threshold.

Next: obtain provider timing/recall clarification. If explicit reference times exist, specify a development-only recall-aligned HR comparison using the same fixed median/Ridge baselines and matched cohorts. No alpha/support search or final-test opening is justified. [Provider questions](../reports/hr_support_loss_provider_questions_draft.md) are drafted only, not sent.

In [ ]:
category_errors = metrics.loc[metrics.stage.eq("loss_only") & metrics.scope.eq("native")
                             & metrics.fold.eq("pooled") & metrics.survey_category.ne("all")
                             & metrics.lookback_minutes.isin([5, 1440])
                             & metrics.model.isin(["median", "context_hr", "q_context_hr"])]
display(category_errors[["lookback_minutes", "survey_category", "model", "responses", "mae", "rmse", "participant_mae"]])
print("Cohort output:", BUNDLE.relative_to(ROOT) / "membership.parquet")
print("Prediction output:", BUNDLE.relative_to(ROOT) / "predictions.parquet")
print("Verified support:", BUNDLE.relative_to(ROOT) / "verified_hr_support.parquet")
print("Aggregate report:", REPORT.relative_to(ROOT) / "report.md")